## 1. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from statistics import NormalDist

tickers = ["TLT", "XLF", "XIU.TO", "SPY", "ZAG.TO", "GLD"]

# Assumptions
PORTFOLIO_VALUE = 10_000_000   # portfolio size in $
RISK_LIMIT = 250_000           # 1-day 99% VaR limit in $
CONF = 0.99
WINDOW = 252                   # rolling window for the backtest (trading days)

prices = yf.download(tickers, start="2019-01-01", auto_adjust=True)["Close"]
prices = prices.dropna()       # keep only dates when all six markets traded
log_returns = np.log(prices / prices.shift(1)).dropna()
print(len(prices), len(log_returns))

## 2. Correlation heatmap and volatility

In [ ]:
corr = log_returns.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Correlation of daily log returns")
fig.show()

vol = (log_returns.std() * np.sqrt(252) * 100).round(1)
print("Annualized volatility (%):")
print(vol.sort_values(ascending=False))

## 3. Portfolio returns (equal weights)

In [ ]:
weights = pd.Series(1 / len(prices.columns), index=prices.columns)

simple_returns = np.exp(log_returns) - 1
port_ret = simple_returns @ weights  

print(f"Portfolio annualized volatility: {port_ret.std() * np.sqrt(252):.1%}")

## 4. VaR and Expected Shortfall (full sample)

In [ ]:
z = NormalDist().inv_cdf(CONF)

hist_var = -np.percentile(port_ret, (1 - CONF) * 100)      # worst 1% of days
param_var = -(port_ret.mean() - z * port_ret.std())        # assumes normal returns
tail = port_ret[port_ret <= -hist_var]
es = -tail.mean()                                          # average loss beyond VaR

var_table = pd.DataFrame({
    "Loss %": [hist_var * 100, param_var * 100, es * 100],
    "Loss $": [hist_var * PORTFOLIO_VALUE, param_var * PORTFOLIO_VALUE, es * PORTFOLIO_VALUE],
}, index=["Historical VaR 99%", "Parametric VaR 99%", "Expected Shortfall 99%"])
print(var_table.round(2))

hist_var_dollars = hist_var * PORTFOLIO_VALUE
print("Historical VaR is within the limit" if hist_var_dollars <= RISK_LIMIT
      else f"Historical VaR BREACHES the ${RISK_LIMIT:,.0f} limit")

fig = px.histogram(x=port_ret * 100, nbins=80,
                   labels={"x": "Daily portfolio return (%)"},
                   title="Distribution of daily portfolio returns")
fig.add_vline(x=-hist_var * 100, line_dash="dash", line_color="red",
              annotation_text="Historical VaR 99%")
fig.show()

## 5. Backtest: rolling VaR vs actual losses

In [ ]:
roll_hist = -port_ret.rolling(WINDOW).quantile(1 - CONF).shift(1)
roll_param = -(port_ret.rolling(WINDOW).mean()
               - z * port_ret.rolling(WINDOW).std()).shift(1)

bt = pd.DataFrame({"loss": -port_ret,
                   "hist_var": roll_hist,
                   "param_var": roll_param}).dropna()

for col in ["hist_var", "param_var"]:
    breaches = bt["loss"] > bt[col]
    print(f"{col}: {breaches.sum()} breaches in {len(bt)} days "
          f"= {breaches.mean():.2%} (expected {1 - CONF:.0%})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=bt.index, y=bt["loss"] * 100, name="Daily loss (%)",
                         mode="lines", line=dict(width=1)))
fig.add_trace(go.Scatter(x=bt.index, y=bt["hist_var"] * 100, name="Rolling historical VaR"))
fig.add_trace(go.Scatter(x=bt.index, y=bt["param_var"] * 100, name="Rolling parametric VaR"))
fig.update_layout(title="Backtest: actual daily loss vs 99% VaR", yaxis_title="%")
fig.show()

## 6. Stress windows: replay on today's portfolio

In [ ]:
windows = {
    "Covid crash (20 Feb - 23 Mar 2020)": ("2020-02-20", "2020-03-23"),
    "2022 rate shock (4 Jan - 12 Oct 2022)": ("2022-01-04", "2022-10-12"),
}

results = {}
full_corr = log_returns["SPY"].corr(log_returns["TLT"])
for name, (start, end) in windows.items():
    seg = simple_returns.loc[start:end]
    asset_ret = (1 + seg).prod() - 1
    asset_ret["PORTFOLIO"] = (1 + seg @ weights).prod() - 1
    results[name] = asset_ret

    window_corr = log_returns.loc[start:end, "SPY"].corr(log_returns.loc[start:end, "TLT"])
    pnl = asset_ret["PORTFOLIO"] * PORTFOLIO_VALUE
    loss_multiple = max(-pnl, 0) / hist_var_dollars
    print(f"{name}")
    print(f"  Portfolio return: {asset_ret['PORTFOLIO']:.1%}  (P&L ${pnl:+,.0f}, "
          f"loss = {loss_multiple:.1f}x one-day VaR)")
    print(f"  SPY-TLT correlation: {window_corr:.2f} in window vs {full_corr:.2f} full sample")

stress = pd.DataFrame(results) * 100
print("\nReturn by asset in each window (%):")
print(stress.round(1))

fig = px.bar(stress.drop("PORTFOLIO"), barmode="group",
             labels={"value": "Return (%)", "index": "Asset", "variable": "Scenario"},
             title="Asset returns in each stress window")
fig.show()

roll_corr = log_returns["SPY"].rolling(60).corr(log_returns["TLT"])
fig = px.line(x=roll_corr.index, y=roll_corr.values,
              labels={"x": "Date", "y": "Correlation"},
              title="60-day rolling correlation: SPY vs TLT")
fig.add_hline(y=0, line_dash="dot")
fig.show()